# 10.3 Reading data from relational tables

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

To use an external relational `table` for reading only, you should employ a `table` declaration
that specifies a read/write status of IN. Thus it should have the general form

```ampl
table table-name IN string-list opt :
    key-spec, data-spec, data-spec, ... ;
```

where the optional string-list is specific to the database type and access method being
used. (In the interest of brevity, most subsequent examples do not show a string-list.)
The key-spec names the key columns, and the `data`-spec gives the `data` columns. Data
values are subsequently read from the `table` into AMPL entities by the command

```ampl
read table table-name ;
```

which determines the values to be read by referring to the `table` declaration that defined
`table`-name.

Reading parameters only
To assign values from `data` columns to like-named AMPL parameters, it suffices to
give a bracketed list of key columns and then a list of `data` columns. The simplest case,
where there is only one key column, is exemplified by

```ampl
table Foods IN: [FOOD], cost, f_min, f_max;
```

This indicates that the relational `table` has four columns, comprising a key column FOOD
and `data` columns cost, f_min and f_max. The `data` columns are associated with
parameters cost, f_min and f_max in the current AMPL `model`. Since there is only
one key column, all of these parameters must be indexed over one-dimensional sets.

When the command

```ampl
read table Foods
```

is executed, the relational `table` is read one row at a time. A row's entry in the key column
is interpreted as a subscript to each of the parameters, and these subscripted parameters
are assigned the row's entries from the associated `data` columns. For example, if the
relational `table` is

```ampl
FOOD    cost    f_min  f_max
BEEF    3.19    2      10
CHK     2.59    2      10
FISH    2.29    2      10
HAM     2.89    2      10
MCH     1.89    2      10
MTL     1.99    2      10
SPG     1.99    2      10
TUR     2.49    2      10
```

processing the first row assigns the values 3.19 to parameter cost['BEEF'], 2 to
f_min['BEEF'], and 10 to f_max['BEEF']; processing the second row assigns 2.59
to cost['CHK'], 2 to f_min['CHK'], and 10 to f_max['CHK']; and so
forth through the six remaining rows.

At the time that the `read table` command is executed, AMPL makes no assumptions
about how the parameters are declared; they need not be indexed over a set named
FOOD, and indeed the members of their indexing sets may not yet even be known. Only
later, when AMPL first uses each parameter in some computation, does it `check` the
entries read from key column FOOD to be sure that each is a valid subscript for that
parameter.

The situation is analogous for multidimensional parameters. The name of each `data`
column must also be the name of an AMPL parameter, and the dimension of the
parameter's indexing set must equal the number of key columns. For example, when two
key columns are listed within the brackets:

```ampl
table SteelProd IN: [PROD, TIME], market, revenue;
```

the listed `data` columns, market and revenue, must correspond to AMPL parameters
market and revenue that are indexed over two-dimensional sets.

When `read table` SteelProd is executed, each row's entries in the key columns
are interpreted as a pair of subscripts to each of the parameters. Thus if the relational
`table` has contents

```ampl
PROD   TIME  market  revenue
bands   1     6000      25
bands   2     6000      26
bands   3     4000      27
bands   4     6500      27
coils   1     4000      30
coils   2     2500      35
coils   3     3500      37
coils   4     4200      39
```

processing the first row will assign 6000 to market['bands',1] and 25 to
revenue['bands',1]; processing the second row will assign 6000 to
market['bands',2] and 26 to revenue['bands',2]; and so forth through all
eight rows. The pairs of subscripts given by the key column entries must be valid for
market and revenue when the values of these parameters are first needed by AMPL,
but the parameters need not be declared over sets named PROD and TIME. (In fact, in the
`model` from which this example is taken, the parameters are indexed by {PROD, 1..T}
where T is a previously defined parameter.)
Since a relational `table` has only one collection of key columns, AMPL applies the
same subscripting to each of the parameters named by the `data` columns. These parameters
are thus usually indexed over the same AMPL set. Parameters indexed over similar
sets may also be accommodated in one database `table`, however, by leaving blank any
entries in rows corresponding to invalid subscripts. The way in which a blank entry is
indicated is specific to the database software being used.

Values of unindexed (scalar) parameters may be supplied by a relational `table` that has
one row and no key columns, so that each `data` column contains precisely one value. The
corresponding `table` declaration has an empty key-spec, \[]. For example, to read a
value for the parameter T that gives the number of periods in steelT.mod, the `table`
declaration is

```ampl
table SteelPeriods IN: [], T;
```

and the corresponding relational `table` has one column, also named T, whose one entry is
a positive `integer`.

Reading a set and parameters
It is often convenient to read the members of a set from a table's key column or
columns, at the same time that parameters indexed over that set are read from the `data`
columns. To indicate that a set should be read from a `table`, the key-spec in the `table`
declaration is written in the form

```ampl
set-name <- [key-col-spec, key-col-spec, ...]
```

The <- symbol is intended as an arrow pointing in the direction that the information is
moved, from the key columns to the AMPL set.

The simplest case involves reading a one-dimensional set and the parameters indexed
over it, as in this example for diet.mod:

```ampl
table Foods IN: FOOD <- [FoodName], cost, f_min, f_max;
```

When the command `read table` Foods is executed, all entries in the key column
FoodName of the relational `table` are read into AMPL as members of the set FOOD, and
the entries in the `data` columns cost, f_min and f_max are read into the like-named
AMPL parameters as previously described. If the key column is named FOOD like the
AMPL set, the appropriate `table` declaration becomes

```ampl
table Foods IN: FOOD <- [FOOD], cost, f_min, f_max;
```

In this special case only, the key-spec can also be written in the abbreviated form
[FOOD] IN.

An analogous syntax is employed for reading a multidimensional set along with
parameters indexed over it. In the case of transp3.mod, for instance, the `table` declaration
could be:

```ampl
table TransLinks IN: LINKS <- [ORIG, DEST], cost;
```

When `read table` TransLinks is executed, each row of the `table` provides a pair of
entries from key columns ORIG and DEST. All such pairs are read into AMPL as members
of the two-dimensional set LINKS. Finally, the entries in column cost are read
into parameter cost in the usual way.

As in our previous multidimensional example, the names in brackets need not correspond
to sets in the AMPL `model`. The bracketed names serve only to identify the key
columns. The name to the left of the arrow is the only one that must name a previously
declared AMPL set; moreover, this set must have been declared to have the same dimension,
or arity, as the number of key columns.

It makes sense to read the set LINKS from a relational `table`, because LINKS is
specifically declared in the `model` in a way that leaves the corresponding `data` to be read
separately:

```ampl
set ORIG;
set DEST;
set LINKS within {ORIG,DEST};
param cost {LINKS} >= 0;
```

By contrast, in the similar `model` transp2.mod, LINKS is defined in terms of two
one-dimensional sets:

```ampl
set ORIG;
set DEST;
set LINKS = {ORIG,DEST};
param cost {LINKS} >= 0;
```

and in transp.mod, no named two-dimensional set is defined at all:

```ampl
set ORIG;
set DEST;
param cost {ORIG,DEST} >= 0;
```

In these latter cases, a `table` declaration would still be needed for reading parameter
cost, but it would not specify the reading of any associated set:

```ampl
table TransLinks IN: [ORIG, DEST], cost;
```

Separate relational tables would instead be used to provide members for the onedimensional
sets ORIG and DEST and values for the parameters indexed over them.

When a `table` declaration specifies an AMPL set to be assigned members, its list of
`data`-specs may be empty. In that case only the key columns are read, and the only action
of `read table` is to assign the key column values as members of the specified AMPL
set. For instance, with the statement

```ampl
table TransLinks IN: LINKS <- [ORIG, DEST];
```

a subsequent `read table` statement would cause just the values for the set LINKS to be
read, from the two key columns in the corresponding database `table`.
**Establishing correspondences**

An AMPL model's set and parameter declarations do not necessarily correspond in all
respects to the organization of tables in relevant databases. Where the difference is substantial,
it may be necessary to use the database's query language (often SQL) to derive
temporary tables that have the structure required by the `model`; an example is given in the
discussion of the ODBC handler later in this chapter. A number of common, simple differences
can be handled directly, however, through features of the `table` declaration.

Differences in naming are perhaps the most common. A `table` declaration can associate
a `data` column with a differently named AMPL parameter by use of a `data`-spec of
the form `param`-name ˜ `data`-col-name. Thus, for example, if `table` Foods were instead
defined by

```ampl
table Foods IN:
   [FOOD], cost, f_min ˜ lowerlim, f_max ˜ upperlim;
```

the AMPL parameters f_min and f_max would be read from `data` columns lowerlim
and upperlim in the relational `table`. (Parameter cost would be read from column
cost as before.)
A similarly generalized form, index ˜ key-col-name, can be used to associate a kind of
dummy index with a key column. This index may then be used in a subscript to the
optional `param`-name in one or more `data`-specs. Such an arrangement is useful in a
number of situations where the key column entries do not exactly correspond to the subscripts
of the parameters that are to receive `table` values. Here are three common cases.

Where a numbering of some kind in the relational `table` is systematically different
from the corresponding numbering in the AMPL `model`, a simple expression involving a
key column index can translate from the one numbering scheme to the other. For example,
if time periods were counted from 0 in the relational `table` `data` rather than from 1 as
in the `model`, an adjustment could be made in the `table` declaration as follows:

```ampl
table SteelProd IN: [p ˜ PROD, t ˜ TIME],
   market[p,t+1] ˜ market, revenue[p,t+1] ˜ revenue;
```

In the second case, where AMPL parameters have subscripts from the same sets but in
different orders, key column indexes must be employed to provide a correct index order.
If market is indexed over {PROD, 1..T} but revenue is indexed over {1..T,
PROD}, for example, a `table` declaration to read values for these two parameters should
be written as follows:

```ampl
table SteelProd IN: [p ˜ PROD, t ˜ TIME],
   market, revenue[t,p] ˜ revenue;
```

Finally, where the values for an AMPL parameter are divided among several database
columns, key column indexes can be employed to describe the values to be found in each
column. For instance, if the revenue values are given in one column for "bands" and
in another column for "coils", the corresponding `table` declaration could be written
like this:

```ampl
table SteelProd IN: [t ˜ TIME],
   revenue["bands",t] ˜ revbands,
   revenue["coils",t] ˜ revcoils;
```

It is tempting to try to shorten declarations of these kinds by dropping the ˜ `data`-colname
, to produce, say,

```ampl
table SteelProd IN:
   [p ˜ PROD, t ˜ TIME], market, revenue[t,p];  # ERROR
```

This will usually be rejected as an error, however, because revenue[t,p] is not a
valid name for a relational `table` column in most database software. Instead it is necessary
to write

```ampl
table SteelProd IN:
   [p ˜ PROD, t ˜ TIME], market, revenue[t,p] ˜ revenue;
```

to indicate that the AMPL parameters revenue[t,p] receive values from the column
revenue of the `table`.

More generally, a ˜ synonym will have to be used in any situation where the AMPL
expression for the recipient of a column's `data` is not itself a valid name for a database
column. The rules for valid column names tend to be the same as the rules for valid component
names in AMPL models, but they can vary in details depending on the database
software that is being used to create and maintain the tables.

**Reading other values**

In a `table` declaration used for input, an assignable AMPL expression may appear
anywhere that a parameter name would be allowed. An expression is assignable if it can
be assigned a value, such as by placing it on the left side of := in a `let` command.

Variable names are assignable expressions. Thus a `table` declaration can specify
columns of `data` to be read into variables, for purposes of evaluating a previously stored
solution or providing a good initial solution for a solver.

Constraint names are also assignable expressions. Values "read into a constraint" are
interpreted as initial dual values for some solvers, such as MINOS.

Any variable or constraint name qualified by an assignable suffix is also an assignable
expression. Assignable suffixes include the predefined .sstatus and .relax as well
as any user-defined suffixes. For example, if the diet problem were changed to have `integer`
variables, the following `table` declaration could help to provide useful information
for the CPLEX solver (see [Section 14.3](#missing) <!--- xTODO missing reference --->):

```ampl
table Foods IN: FOOD IN,
   cost, f_min, f_max, Buy, Buy.priority ˜ prior;
```

An execution of `read table` Foods would supply members for set FOOD and values
for parameters cost, f_min and f_max in the usual way, and would also assign initial
values and branching priorities to the Buy variables.